# Notebook that retrieves API data related to tracks

## Imports

In [ ]:
from _spo_utils import camel_to_snake, chunk_list

from base64 import b64encode
from functools import reduce
import pandas as pd
import requests

# Constants

In [ ]:
PATH_SPOTIFY = '../../data/2_processed/final_df.csv'

## Reading dataset

In [ ]:
df_treated = pd.read_csv(PATH_SPOTIFY)
unique_track_id_list = df_treated['track_id'].unique()

## ReccoBeats APIs - [Get multiple track](https://reccobeats.com/docs/apis/get-tracks) and [Get multiple audio features](https://reccobeats.com/docs/apis/get-audio-features)

In [ ]:
headers = {'Accept': 'application/json'}

endpoints = {
    'audio_features': 'https://api.reccobeats.com/v1/audio-features?ids=',
    'multiple_track': 'https://api.reccobeats.com/v1/track?ids=',
}

frames = {key: [] for key in endpoints}

for chunk in chunk_list(data=unique_track_id_list, chunk_size=40):
    print(f'Len: {len(chunk)}')
    ids_str = ','.join(chunk)

    for key, base_url in endpoints.items():
        resp = requests.get(url=base_url + ids_str, headers=headers)
        frames[key].append(pd.json_normalize(resp.json(), record_path='content'))

df_audio_features = pd.concat(frames['audio_features'], ignore_index=True)
df_multiple_track = pd.concat(frames['multiple_track'], ignore_index=True)

df_audio_features['track_id'] = df_audio_features['href'].str.rpartition('/')[2]
df_multiple_track['track_id'] = df_multiple_track['href'].str.rpartition('/')[2]

## Checking results

In [5]:
display(df_audio_features.head(5))
display(df_multiple_track.head(5))

,id,href,isrc,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,track_id
0,666543a0-04fb-40c1-845b-668beefa1a88,https://open.spotify.com/track/6RUKPb4LETWmmr3...,USQX91700278,0.04980,0.617,0.635,0.000014,11.0,0.1640,-6.769,0.0,0.0317,103.019,0.446,6RUKPb4LETWmmr3iAEQktW
1,96758562-28d9-43de-9249-79a49b8f4994,https://open.spotify.com/track/4tHqQMWSqmL6YjX...,USSM19912814,0.99400,0.418,0.106,0.029200,8.0,0.1790,-22.507,0.0,0.0448,46.718,0.800,4tHqQMWSqmL6YjXwsqthDI
2,80dd06dd-8508-4ac0-b023-d21c40fe2c5a,https://open.spotify.com/track/0HmONWWIU1FXkwW...,NOG841549020,0.00073,0.750,0.617,0.318000,6.0,0.1050,-6.725,1.0,0.0318,89.968,0.108,0HmONWWIU1FXkwWLDpqrjl
3,59935bea-5e0a-4c00-8269-b01baa8c1992,https://open.spotify.com/track/1dNIEtp7AY3oDAK...,USQX91700278,0.03060,0.607,0.649,0.000025,11.0,0.1740,-6.695,0.0,0.0362,102.996,0.505,1dNIEtp7AY3oDAKCGg2XkH
4,ca0b1c69-be5d-4754-a490-efba6d4911fc,https://open.spotify.com/track/0u8aj0c4IxeVSLT...,BRSME1700003,0.37400,0.854,0.682,0.000000,9.0,0.0697,-5.818,0.0,0.0690,98.023,0.887,0u8aj0c4IxeVSLTUuuq9V5


,id,trackTitle,artists,durationMs,isrc,ean,upc,href,availableCountries,popularity,track_id
0,666543a0-04fb-40c1-845b-668beefa1a88,Something Just Like This,[{'id': 'e074334e-20b0-4f9e-a212-5b5632a4e935'...,247160,USQX91700278,None,None,https://open.spotify.com/track/6RUKPb4LETWmmr3...,"AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,D...",86,6RUKPb4LETWmmr3iAEQktW
1,96758562-28d9-43de-9249-79a49b8f4994,Carol of the Bells,[{'id': '640564f3-58c4-4b17-bd41-e228e77bf7c1'...,85266,USSM19912814,None,None,https://open.spotify.com/track/4tHqQMWSqmL6YjX...,"AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,D...",80,4tHqQMWSqmL6YjXwsqthDI
2,80dd06dd-8508-4ac0-b023-d21c40fe2c5a,Faded - Instrumental,[{'id': '0a5d81f9-b98b-46b7-84e7-7b19f340cf61'...,214013,NOG841549020,None,None,https://open.spotify.com/track/0HmONWWIU1FXkwW...,"AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,D...",45,0HmONWWIU1FXkwWLDpqrjl
3,59935bea-5e0a-4c00-8269-b01baa8c1992,Something Just Like This,[{'id': 'e074334e-20b0-4f9e-a212-5b5632a4e935'...,247626,USQX91700278,None,None,https://open.spotify.com/track/1dNIEtp7AY3oDAK...,"AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,D...",69,1dNIEtp7AY3oDAKCGg2XkH
4,ca0b1c69-be5d-4754-a490-efba6d4911fc,Você Partiu Meu Coração (feat. Anitta & Wesley...,[{'id': 'dd7180b0-f357-4ef8-9ea4-9861c91f4001'...,179000,BRSME1700003,None,None,https://open.spotify.com/track/0u8aj0c4IxeVSLT...,"AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,D...",54,0u8aj0c4IxeVSLTUuuq9V5


## Merging dataframes

In [ ]:
# Columns we want to bring in from each auxiliary DataFrame
audio_cols = [
    'track_id', 
    'acousticness', 
    'danceability', 
    'energy',
    'instrumentalness', 
    'key', 
    'liveness', 
    'loudness',
    'mode', 
    'speechiness', 
    'tempo', 
    'valence'
]

track_cols = ['track_id', 'popularity', 'durationMs']

dfs_to_merge = [
    df_audio_features[audio_cols],
    df_multiple_track[track_cols],
]

df_final = reduce(
    lambda left, right: left.merge(right, on='track_id', how='left'),
    dfs_to_merge,
    df_treated.copy() # start with the treated DataFrame
)

### Renaming columns to fit snake_case pattern

In [ ]:
df_final.columns = camel_to_snake(df_final.columns)

In [18]:
display(df_final.head(5))

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,...,key,liveness,loudness,mode,speechiness,tempo,valence,popularity,duration_ms,percentage_played
0,2017-11-21T11:46:01Z,"iOS 9.3.5 (iPad2,5)",6826,BR,201.6.225.101,The Society,Injustice 2: Original Video Game Soundtrack,spotify:track:74ovIDtL0HzDazMENhR0yX,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-11-21T11:46:18Z,"iOS 9.3.5 (iPad2,5)",16648,BR,201.6.225.101,Injustice 2 Main Theme,Injustice 2: Original Video Game Soundtrack,spotify:track:0QKHV9dERThRvpdzQWy8qd,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-11-21T11:46:19Z,"iOS 9.3.5 (iPad2,5)",766,BR,201.6.225.101,Brainiac Takes Krypton,Injustice 2: Original Video Game Soundtrack,spotify:track:7urxa3s1BTWXmx2Eb9QWLp,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-11-21T11:48:59Z,"iOS 9.3.5 (iPad2,5)",15371,BR,201.6.225.101,Gotham Projection Room,Injustice 2: Original Video Game Soundtrack,spotify:track:6qNM4ljTc8PGO3hzjZj4sg,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-11-21T11:50:16Z,"iOS 9.3.5 (iPad2,5)",58653,BR,201.6.225.101,Waiting for Superman,Baptized (Deluxe Version),spotify:track:4AU7z13HYmPMetlWbq1mys,NaN,NaN,...,0.0,0.0662,-5.711,1.0,0.0269,105.987,0.383,56.0,266960.0,0.2197


In [ ]:
df_final.to_csv(PATH_SPOTIFY, index=False)